In [ ]:
# [Benchmark] Standardized Import & Setup
import os
import time
import tracemalloc
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    adjusted_rand_score, 
    normalized_mutual_info_score, 
    adjusted_mutual_info_score, 
    homogeneity_score, 
    completeness_score, 
    v_measure_score,
    f1_score,
    pairwise,
    silhouette_score
)
from scipy.optimize import linear_sum_assignment
from sklearn.neighbors import kneighbors_graph

# NicheCompass Imports
from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata, \
                               extract_gp_dict_from_omnipath_lr_interactions, \
                               extract_gp_dict_from_mebocost_ms_interactions, \
                               filter_and_combine_gp_dict_gps_v2)
import squidpy as sq

warnings.filterwarnings("ignore")

# Plotting Aesthetics
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
sc.settings.verbosity = 0

In [ ]:
# [Utils] Standardization Functions (Plotting & Metrics)

def calculate_metrics(adata, pred_key, true_key, resource_stats, emb_key='latent_embedding'):
    """Calculate clustering metrics and output performance statistics."""
    metrics = {}
    
    # 1. Ground Truth Metrics
    if true_key in adata.obs.columns:
        labels_true = adata.obs[true_key]
        labels_pred = adata.obs[pred_key]
        
        # Standard Clustering Metrics
        metrics['ARI'] = adjusted_rand_score(labels_true, labels_pred)
        metrics['NMI'] = normalized_mutual_info_score(labels_true, labels_pred)
        metrics['AMI'] = adjusted_mutual_info_score(labels_true, labels_pred)
        metrics['Homogeneity'] = homogeneity_score(labels_true, labels_pred)
        metrics['Completeness'] = completeness_score(labels_true, labels_pred)
        metrics['V-Measure'] = v_measure_score(labels_true, labels_pred)

        # Optimal Matching for Classification Metrics (Macro-F1 & Composition Similarity)
        ct = pd.crosstab(labels_true, labels_pred)
        row_ind, col_ind = linear_sum_assignment(-ct.values)
        map_pred_to_true = {}
        for r, c in zip(row_ind, col_ind):
            map_pred_to_true[ct.columns[c]] = ct.index[r]
        mapped_preds = labels_pred.map(map_pred_to_true)
        if mapped_preds.isna().any():
            mapped_preds = mapped_preds.astype(object).fillna("Unmapped").astype(str)

        # Macro-F1 Score (using mapped labels)
        true_cats = labels_true.astype('category').cat.categories
        metrics['Macro-F1'] = f1_score(labels_true, mapped_preds, labels=true_cats, average='macro', zero_division=0)

        # Composition Similarity (using mapped labels)
        if 'cell_type' in adata.obs:
            gt_comp = pd.crosstab(adata.obs[true_key], adata.obs['cell_type'], normalize='index')
            pred_comp = pd.crosstab(mapped_preds, adata.obs['cell_type'], normalize='index')
            common_niches = gt_comp.index.intersection(pred_comp.index)
            if len(common_niches) > 0:
                gt_vecs = gt_comp.loc[common_niches].values
                pred_vecs = pred_comp.loc[common_niches].values
                cos_sims = np.diag(pairwise.cosine_similarity(gt_vecs, pred_vecs))
                metrics['Composition Similarity'] = np.mean(cos_sims)
            else:
                metrics['Composition Similarity'] = 0.0
        else:
             metrics['Composition Similarity'] = 0.0
    else:
        print("Warning: Ground truth key not found. Supervised metrics set to 0.")
        metrics = {k: 0.0 for k in ['ARI', 'NMI', 'AMI', 'Homogeneity', 'Completeness', 'V-Measure', 'Macro-F1', 'Composition Similarity']}
    
    # Spatial Connectivity
    try:
        if 'spatial_connectivities' not in adata.obsp:
            coords = adata.obsm['spatial'] if 'spatial' in adata.obsm else adata.obs[['x', 'y']].values
            adj = kneighbors_graph(coords, n_neighbors=6, mode='connectivity', include_self=False)
            adata.obsp['spatial_connectivities'] = adj
        W_conn = adata.obsp['spatial_connectivities']

        # Vectorized computation
        if hasattr(W_conn, 'indices'):
            # Convert to one-hot (N x K)
            y_onehot = pd.get_dummies(labels_pred).values
            # Neighbor counts per class: (N x N) @ (N x K) -> (N x K)
            neighbor_counts = W_conn @ y_onehot
            # Get count of neighbors with same label
            label_indices = np.argmax(y_onehot, axis=1)
            same_counts = neighbor_counts[np.arange(len(label_indices)), label_indices]
            
            degrees = np.array(W_conn.sum(axis=1)).flatten()
            mask = degrees > 0
            if mask.any():
                metrics['Connectivity'] = float(np.mean(same_counts[mask] / degrees[mask]))
            else:
                metrics['Connectivity'] = 0.0
        else:
            metrics['Connectivity'] = 0.0
    except Exception:
        metrics['Connectivity'] = 0.0

    # Silhouette Score (Latent Embedding)
    if emb_key in adata.obsm:
        try:
            label_key = true_key if true_key in adata.obs else pred_key
            metrics['Silhouette Score'] = silhouette_score(adata.obsm[emb_key], adata.obs[label_key])
        except Exception as e:
            print(f"Warning: Silhouette Score calculation failed: {e}")
            metrics['Silhouette Score'] = 0.0
    else:
        print(f"Warning: Latent embedding key '{emb_key}' not found in adata.obsm. Silhouette Score set to 0.")
        metrics['Silhouette Score'] = 0.0

    print(f"{'='*40}")
    print(f"BENCHMARK METRICS REPORT")
    print(f"{'='*40}")
    for k, v in metrics.items():
        print(f"{k:<20}: {v:.4f}")
        
    print(f"{'-'*40}")
    print(f"Time (s)            : {resource_stats['time']:.4f}")
    print(f"Memory (MB)         : {resource_stats['memory']:.4f}")
    print(f"Memory Peak (MB)    : {resource_stats['memory_peak']:.4f}")
    print(f"{'='*40}")
    
    return metrics

def plot_benchmark_results(adata, pred_key, true_key, cell_type_key, emb_key='latent_embedding'):
    """Standardized visualization: Ground Truth, Prediction, and Cell Type Composition."""
    
    for key in [pred_key, true_key, cell_type_key]:
        if key in adata.obs and not pd.api.types.is_categorical_dtype(adata.obs[key]):
            adata.obs[key] = adata.obs[key].astype('category')

    # A. Ground Truth Map
    print("Plotting Ground Truth Niche...")
    if true_key in adata.obs:
        if f'{true_key}_colors' in adata.uns:
            del adata.uns[f'{true_key}_colors']
            
        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.scatter(adata, x='x', y='y', color=true_key, 
                     title='Ground Truth Niche', size=120000/adata.n_obs, frameon=False, show=False, ax=ax)
        ax.set_aspect('equal', 'box')
        plt.show()
    else:
        plt.figure(figsize=(6, 6))
        plt.text(0.5, 0.5, 'Ground Truth Not Available', ha='center', va='center')
        plt.axis('off')
        plt.show()

    # B. Predicted Map
    print("Plotting Predicted Niche...")
    if f'{pred_key}_colors' in adata.uns:
        del adata.uns[f'{pred_key}_colors']
        
    fig, ax = plt.subplots(figsize=(6, 6))
    sc.pl.scatter(adata, x='x', y='y', color=pred_key, 
                 title='Predicted Niche', size=120000/adata.n_obs, frameon=False, show=False, ax=ax)
    ax.set_aspect('equal', 'box')
    plt.show()

    # C. Latent Embedding UMAP
    print("Plotting Latent Embedding UMAP...")
    
    if emb_key in adata.obsm:
        sc.pp.neighbors(adata, use_rep=emb_key, n_neighbors=30)
        sc.tl.umap(adata)
        
        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.umap(adata, color=pred_key, title=f'Latent Embedding UMAP', 
                   frameon=False, show=False, ax=ax)
        plt.show()
    else:
        print(f"Latent embedding key '{emb_key}' not found in adata.obsm")

    # D. Composition Analysis
    print("Plotting Composition Analysis...")
    
    if f'{cell_type_key}_colors' not in adata.uns:
        from scanpy.plotting import palettes
        length = len(adata.obs[cell_type_key].cat.categories)
        if length <= 20:
            palette = palettes.default_20
        else:
            palette = palettes.default_102
        adata.uns[f'{cell_type_key}_colors'] = palette[:length]
    
    ctype_colors = adata.uns[f'{cell_type_key}_colors']
    ctype_cats = adata.obs[cell_type_key].cat.categories
    ctype_cmap = dict(zip(ctype_cats, ctype_colors))

    df = pd.DataFrame({'pred': adata.obs[pred_key], 'cell_type': adata.obs[cell_type_key]})
    comp = pd.crosstab(df['pred'], df['cell_type'], normalize='index')
    
    bar_colors = [ctype_cmap.get(c, 'grey') for c in comp.columns]
    
    plt.figure(figsize=(10, 6))
    ax = plt.gca()
    comp.plot(kind='bar', stacked=True, width=0.8, ax=ax, color=bar_colors)
    
    ax.set_title('Cell Type Composition per Predicted Niche')
    ax.set_xlabel('Predicted Niche')
    ax.set_ylabel('Fraction')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title='Cell Type')
    
    plt.tight_layout()
    plt.show()

def save_benchmark_results(adata, output_path, method_name, pred_key, emb_key):
    """
    Save benchmark results to a persistent h5ad file.
    Appends/Updates 'pred_niche_{method_name}' and 'X_{method_name}'.
    """
    print(f"[Save] Saving results for '{method_name}' to {output_path}...")
    
    target_pred = f"pred_niche_{method_name}"
    target_emb = f"X_{method_name}"
    
    if os.path.exists(output_path):
        print(f"  - Loading existing data from {output_path}...")
        adata_out = sc.read_h5ad(output_path)
    else:
        print(f"  - File not found. Creating new dataset from current results...")
        adata_out = adata.copy()       
        keys_to_keep = [k for k in adata_out.uns.keys() if 'colors' in k]
        adata_out.uns = {k: adata_out.uns[k] for k in keys_to_keep}
    
    if pred_key in adata.obs:
        print(f"  - Updating obs['{target_pred}']")
        adata_out.obs[target_pred] = adata.obs[pred_key]
    else:
        print(f"  - Warning: Prediction key '{pred_key}' not found in current adata.")

    if emb_key in adata.obsm:
        print(f"  - Updating obsm['{target_emb}']")
        if adata_out.shape[0] == adata.shape[0]:
            adata_out.obsm[target_emb] = adata.obsm[emb_key]
        else:
            print(f"  - Error: Shape mismatch for embedding (Out: {adata_out.shape}, In: {adata.shape}). Skipping embedding save.")
    else:
        print(f"  - Warning: Embedding key '{emb_key}' not found in current adata.")

    print(f"  - Writing to disk...")
    try:
        adata_out.write_h5ad(output_path)
        print(f"  - Save complete.")
        print(adata_out)
    except Exception as e:
        print(f"  - Error saving file: {e}")

def save_time_memory_results(stats, method_name, file_path='./time_memory.csv'):
    """Save time and memory stats to CSV, updating if method exists."""
    data = {
        'Method': method_name,
        'Time': stats['time'],
        'Memory': stats['memory'],
        'Memory_Peak': stats['memory_peak']
    }
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        if method_name in df['Method'].values:
            for key, value in data.items():
                if key != 'Method':
                    df.loc[df['Method'] == method_name, key] = value
        else:
            df = pd.concat([df, pd.DataFrame([data])], ignore_index=True)
    else:
        df = pd.DataFrame([data])
        
    df.to_csv(file_path, index=False)
    print(f"[Save] Resource stats saved to {file_path}")

In [ ]:
def run_nichecompass_method(adata, n_clusters=4):
    # 1. Setup Tracking
    start_time = time.time()
    tracemalloc.start()
    
    if adata.X.dtype != 'float32':
        adata.X = adata.X.astype('float32')

    from scipy import sparse
    if not sparse.issparse(adata.X):
        print("[NicheCompass] Converting adata.X to sparse CSR matrix for compatibility...")
        adata.X = sparse.csr_matrix(adata.X)
    
    if 'counts' not in adata.layers:
        adata.layers['counts'] = adata.X.copy()
    elif adata.layers['counts'].dtype != 'float32':
        adata.layers['counts'] = adata.layers['counts'].astype('float32')

    if not sparse.issparse(adata.layers['counts']):
        adata.layers['counts'] = sparse.csr_matrix(adata.layers['counts'])
    
    # Data Preparation Keys
    counts_key = "counts"
    adj_key = "spatial_connectivities"
    gp_names_key = "nichecompass_gp_names"
    active_gp_names_key = "nichecompass_active_gp_names"
    gp_targets_mask_key = "nichecompass_gp_targets"
    gp_targets_categories_mask_key = "nichecompass_gp_targets_categories"
    gp_sources_mask_key = "nichecompass_gp_sources"
    gp_sources_categories_mask_key = "nichecompass_gp_sources_categories"
    latent_key = "nichecompass_latent"
    latent_cluster_key = "latent_niche"
    
    if adj_key not in adata.obsp:
        sq.gr.spatial_neighbors(adata, coord_type="generic", spatial_key="spatial", n_neighs=10)
        # Symmetrize
        adata.obsp[adj_key] = (adata.obsp[adj_key].maximum(adata.obsp[adj_key].T))
    
    # 2. Gene Program Extraction
    gp_dicts_list = []
    
    # A. Omnipath
    omnipath_path = "./nichecompass_gp/omnipath_lr_network.csv"
    
    if os.path.exists(omnipath_path):
        try:
            df_omni = pd.read_csv(omnipath_path, index_col=0)
            crit_cols = ["genesymbol_intercell_source", "genesymbol_intercell_target"]
            if all(col in df_omni.columns for col in crit_cols):
                df_clean = df_omni.dropna(subset=crit_cols)
                temp_path = "./temp_omnipath_lr_network.csv"
                df_clean.to_csv(temp_path)
                use_path = temp_path
            else:
                print("[NicheCompass] Warning: Expected columns not found in Omnipath CSV. Using original.")
                use_path = omnipath_path
        except Exception as e:
            print(f"[NicheCompass] Warning: Failed to clean Omnipath CSV: {e}. Using original.")
            use_path = omnipath_path

        print(f"[NicheCompass] Loading Omnipath GP dict from {use_path}...")
        try:
            gp_dict = extract_gp_dict_from_omnipath_lr_interactions(
                species="human",
                min_curation_effort=0,
                load_from_disk=True,
                lr_network_file_path=use_path
            )
            gp_dicts_list.append(gp_dict)
        except Exception as e:
             print(f"[NicheCompass] Error loading Omnipath GPs: {e}")
        if use_path != omnipath_path and os.path.exists(use_path):
            try:
                os.remove(use_path)
            except:
                pass
    else:
        print(f"Warning: Omnipath GP file not found at {omnipath_path}.")

    # B. MEBOCOST (Metabolite Enzyme-Sensor)
    mebocost_path = "./nichecompass_gp/metabolite_enzyme_sensor_gps"
    if os.path.exists(mebocost_path):
        print(f"[NicheCompass] Loading MEBOCOST GP dict from {mebocost_path}...")
        try:
            mebocost_gp_dict = extract_gp_dict_from_mebocost_ms_interactions(
                dir_path=mebocost_path,
                species="human"
            )
            gp_dicts_list.append(mebocost_gp_dict)
        except Exception as e:
            print(f"[NicheCompass] Warning: Failed to load MEBOCOST GPs: {e}")
    else:
        print(f"Warning: MEBOCOST folder not found at {mebocost_path}")

    # Combine and Add GPs
    if len(gp_dicts_list) > 0:
        print(f"[NicheCompass] Combining {len(gp_dicts_list)} GP dictionaries...")
        combined_gp_dict = filter_and_combine_gp_dict_gps_v2(
            gp_dicts=gp_dicts_list,
            verbose=False,
        )
        
        add_gps_from_gp_dict_to_adata(
            gp_dict=combined_gp_dict,
            adata=adata,
            gp_targets_mask_key=gp_targets_mask_key,
            gp_targets_categories_mask_key=gp_targets_categories_mask_key,
            gp_sources_mask_key=gp_sources_mask_key,
            gp_sources_categories_mask_key=gp_sources_categories_mask_key,
            gp_names_key=gp_names_key,
            min_genes_per_gp=2,
            min_source_genes_per_gp=1,
            min_target_genes_per_gp=1,
            max_genes_per_gp=None,
            max_source_genes_per_gp=None,
            max_target_genes_per_gp=None
        )
    else:
        print("Warning: No Gene Programs loaded. Running without specific GPs.")
    
    # 3. Model Training
    print("[NicheCompass] Initializing and training model...")
    model = NicheCompass(
        adata,
        counts_key=counts_key,
        adj_key=adj_key,
        gp_names_key=gp_names_key,
        active_gp_names_key=active_gp_names_key,
        gp_targets_mask_key=gp_targets_mask_key,
        gp_targets_categories_mask_key=gp_targets_categories_mask_key,
        gp_sources_mask_key=gp_sources_mask_key,
        gp_sources_categories_mask_key=gp_sources_categories_mask_key,
        latent_key=latent_key,
        conv_layer_encoder="gcnconv",
        active_gp_thresh_ratio=0.01,
    )
    
    n_epochs = 100
    model.train(
        n_epochs=n_epochs,
        lr=0.001,
        lambda_edge_recon=500000.,
        lambda_gene_expr_recon=300.,
        lambda_l1_masked=0.,
        edge_batch_size=1024,
        n_sampled_neighbors=10,
        use_cuda_if_available=True,
        verbose=True
    )
    
    # 4. Post-processing with Target K optimization
    print("[NicheCompass] Computing latent clusters...")
    sc.pp.neighbors(model.adata, use_rep=latent_key, key_added=latent_key)
    
    print(f"[NicheCompass] Searching resolution for K={n_clusters}...")
    
    def find_resolution(adata, target_k, lower=0.01, upper=1.0, steps=40, max_rounds=6):
        """Search resolution in [lower, upper] until a run yields exactly K=target_k."""
        best_res = None
        
        cur_lower = float(lower)
        cur_upper = float(upper)

        for round_idx in range(max_rounds):
            resolutions = np.linspace(cur_lower, cur_upper, steps)
            print(f"  [find_resolution] Round {round_idx+1}/{max_rounds}: scan [{cur_lower:.4f}, {cur_upper:.4f}]")
            
            found = False
            for res in resolutions:
                res = float(res)
                try:
                    sc.tl.leiden(adata, resolution=res, key_added='temp_leiden', neighbors_key=latent_key)
                    n_found = len(adata.obs['temp_leiden'].unique())
                    
                    if n_found == target_k:
                        best_res = res
                        print(f"    Found K={target_k} at res={res:.4f}")
                        found = True
                        break
                except Exception:
                    pass
            
            if found:
                break
                
            span = cur_upper - cur_lower
            if span <= 0: span = 0.1
            cur_lower = max(0.0, cur_lower - span)
            cur_upper = cur_upper + span

        if 'temp_leiden' in adata.obs:
            try:
                del adata.obs['temp_leiden']
            except Exception:
                pass

        return best_res, (best_res is not None)

    # Execute search
    optimal_res, found_exact_k = find_resolution(model.adata, n_clusters)

    if found_exact_k:
        sc.tl.leiden(
            adata=model.adata,
            resolution=optimal_res,
            key_added=latent_cluster_key,
            neighbors_key=latent_key
        )
    else:
        from sklearn.cluster import KMeans

        X = model.adata.obsm[latent_key]
        print(f"  [Fallback] KMeans used to guarantee K={n_clusters}")
        km = KMeans(n_clusters=n_clusters, n_init='auto', random_state=0)
        final_labels = km.fit_predict(X)
        model.adata.obs[latent_cluster_key] = pd.Series(final_labels, index=model.adata.obs_names).astype(str)
    
    adata.obs['pred_niche'] = model.adata.obs[latent_cluster_key]
    if latent_key in model.adata.obsm:
        adata.obsm['latent_embedding'] = model.adata.obsm[latent_key]
    else:
        print(f"Warning: Latent key '{latent_key}' not found in model.adata.obsm")

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end_time = time.time()
    
    stats = {
        "time": end_time - start_time,
        "memory": current / 10**6,
        "memory_peak": peak / 10**6,
        "epochs": n_epochs
    }
    
    return adata, stats

In [ ]:
# [Main] Execution

if __name__ == "__main__":
    tasks = [
        ("./lymph_node_niche_annotated.h5ad", "./lymph_node_niche_results.h5ad", "./time_memory.csv")
    ]
    N_CLUSTERS = 4
    
    for FILE_PATH, OUTPUT_PATH, TIMEMEM_PATH in tasks:
        print(f"\n[{'='*30} Processing {FILE_PATH} {'='*30}]")
        
        # 1. Load Data
        if not os.path.exists(FILE_PATH):
            print(f"Data file not found: {FILE_PATH}, skipping...")
            continue
        
        print(f"[Data] Loading {FILE_PATH}...")
        adata = sc.read_h5ad(FILE_PATH)
        
        if 'x' not in adata.obs.columns:
            if 'spatial' in adata.obsm:
                adata.obs['x'] = adata.obsm['spatial'][:, 0]
                adata.obs['y'] = adata.obsm['spatial'][:, 1]
            else:
                print("Warning: No spatial coordinates found.")

        print(f"[Data] Shape: {adata.shape}")
        if 'niche' in adata.obs:
            print(f"[Data] Ground Truth Niches: {len(adata.obs['niche'].unique())}")
    
        # 2. Run Method
        print(f"[Benchmark] Running NicheCompass Algorithm...")
        adata, stats = run_nichecompass_method(adata, n_clusters=N_CLUSTERS)
        
        # 3. Report & Visualize
        calculate_metrics(adata, pred_key='pred_niche', true_key='niche', resource_stats=stats, emb_key='latent_embedding')
        plot_benchmark_results(adata, pred_key='pred_niche', true_key='niche', cell_type_key='cell_type', emb_key='latent_embedding')

        # 4. Save Results
        save_benchmark_results(adata, OUTPUT_PATH, method_name='NicheCompass', pred_key='pred_niche', emb_key='latent_embedding')
        
        # 5. Save Time & Memory
        save_time_memory_results(stats, method_name='NicheCompass', file_path=TIMEMEM_PATH)